# 1. Audio Download & Speech Classification

Three independent, resumable stages, each checkpointed to its own CSV so a failure in one doesn't force redoing the others:

1. **Export** — query the DB once into `data/recordings.csv` (cached; reruns reuse it unless `FORCE_REFRESH_DB=1`).
2. **Download** — read `recordings.csv`, download each recording's audio into `data/audio/{id}.wav` (gitignored) via `ffmpeg`. Skips files already on disk. Writes to a `.partial` temp file first and only renames it into place on success, so an interrupted download never masquerades as a completed one on the next run.
3. **Classify** — read `recordings.csv`, run Silero VAD on whatever audio has been downloaded, and append the original DB columns plus a single `is_speech` (boolean) column to `data/audio_speech_labels.csv`. Rows that fail or have no downloaded audio are skipped and logged, not written.

In [ ]:
import os
import subprocess
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import pandas as pd
import psycopg
import torch
from dotenv import load_dotenv
from scipy.io import wavfile
from silero_vad import get_speech_timestamps, load_silero_vad
from tqdm import tqdm

load_dotenv()

## Configuration

All values come from `.env` at the repo root (found automatically by `load_dotenv()` walking up from this notebook's directory), with sensible fallbacks.

In [ ]:
# Database
DB_HOST = os.getenv("DB_HOST")
DB_PORT = int(os.getenv("DB_PORT", "5432"))
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
TABLE_NAME = os.getenv("TABLE_NAME")

# Name of the column containing the streamable audio URL.
AUDIO_URL_COLUMN = os.getenv("AUDIO_URL_COLUMN", "streamableUrl")

# Name of the row identifier column, used for resuming and for naming
# downloaded audio files on disk.
ID_COLUMN = os.getenv("ID_COLUMN", "id")

_where_clause_env = os.getenv("WHERE_CLAUSE", "").strip()
WHERE_CLAUSE = _where_clause_env if _where_clause_env else None

_limit_env = os.getenv("LIMIT", "").strip()
LIMIT = int(_limit_env) if _limit_env else None

# Re-query the DB even if a cached recordings.csv already exists.
FORCE_REFRESH_DB = os.getenv("FORCE_REFRESH_DB", "").strip().lower() in {"1", "true", "yes"}

# Output locations (relative to this notebook's directory, i.e. jupyter_notebooks/)
DATA_DIR = Path(os.getenv("DATA_DIR", "../data"))
AUDIO_DIR = DATA_DIR / "audio"
RECORDINGS_CSV = DATA_DIR / "recordings.csv"
OUTPUT_CSV = DATA_DIR / os.getenv("OUTPUT_CSV", "audio_speech_labels.csv")
SAVE_EVERY = int(os.getenv("SAVE_EVERY", "10"))

DATA_DIR.mkdir(parents=True, exist_ok=True)
AUDIO_DIR.mkdir(parents=True, exist_ok=True)

# Speech detection
SAMPLE_RATE = int(os.getenv("SAMPLE_RATE", "16000"))

# A recording is labelled as speech when at least this proportion of it
# contains detected speech.
SPEECH_RATIO_THRESHOLD = float(os.getenv("SPEECH_RATIO_THRESHOLD", "0.20"))

MIN_SPEECH_DURATION_MS = int(os.getenv("MIN_SPEECH_DURATION_MS", "250"))
MIN_SILENCE_DURATION_MS = int(os.getenv("MIN_SILENCE_DURATION_MS", "300"))
SPEECH_PAD_MS = int(os.getenv("SPEECH_PAD_MS", "100"))
DOWNLOAD_TIMEOUT_SECONDS = int(os.getenv("DOWNLOAD_TIMEOUT_SECONDS", "180"))

# Number of parallel workers. Each gets its own Silero model instance during
# the classify stage.
MAX_WORKERS = int(os.getenv("MAX_WORKERS", "8"))

torch.set_num_threads(1)

## Stage 1 — Export DB rows to `recordings.csv`

In [ ]:
def connect_to_database():
    return psycopg.connect(
        host=DB_HOST,
        port=DB_PORT,
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD,
    )


def load_database_rows() -> pd.DataFrame:
    query = f"SELECT * FROM {TABLE_NAME}"

    if WHERE_CLAUSE:
        query += f" WHERE {WHERE_CLAUSE}"

    if LIMIT is not None:
        query += f" LIMIT {int(LIMIT)}"

    print("Reading rows from PostgreSQL...")

    with connect_to_database() as connection:
        with connection.cursor() as cursor:
            cursor.execute(query)
            column_names = [description.name for description in cursor.description]
            rows = cursor.fetchall()

    dataframe = pd.DataFrame(rows, columns=column_names)

    print(f"Loaded {len(dataframe)} rows.")

    return dataframe

In [ ]:
if RECORDINGS_CSV.exists() and not FORCE_REFRESH_DB:
    recordings = pd.read_csv(RECORDINGS_CSV)
    print(
        f"Loaded {len(recordings)} cached rows from {RECORDINGS_CSV}.\n"
        "Set FORCE_REFRESH_DB=1 in .env to re-query the DB instead."
    )
else:
    recordings = load_database_rows()

    if AUDIO_URL_COLUMN not in recordings.columns:
        raise ValueError(
            f"Column '{AUDIO_URL_COLUMN}' was not found.\n"
            f"Available columns: {list(recordings.columns)}"
        )

    if ID_COLUMN not in recordings.columns:
        raise ValueError(
            f"Column '{ID_COLUMN}' was not found.\n"
            f"Available columns: {list(recordings.columns)}"
        )

    recordings.to_csv(RECORDINGS_CSV, index=False)
    print(f"Saved {len(recordings)} rows to {RECORDINGS_CSV}")

## Stage 2 — Download audio (from `recordings.csv`)

In [ ]:
def download_audio(
    url: str,
    destination_path: Path,
    sample_rate: int = SAMPLE_RATE,
    timeout_seconds: int = DOWNLOAD_TIMEOUT_SECONDS,
) -> None:
    """Download and decode `url` into a mono WAV file at `destination_path` via ffmpeg.

    Writes to a temporary `.partial` path first and only renames it into place on
    success, so a failed/interrupted download never leaves a file behind that a
    later run would mistake for an already-downloaded one. The output format is
    forced with `-f wav` since ffmpeg can't infer it from the `.partial` extension.
    """
    temporary_path = destination_path.with_suffix(destination_path.suffix + ".partial")

    command = [
        "ffmpeg",
        "-y",
        "-loglevel", "error",
        "-rw_timeout", str(timeout_seconds * 1_000_000),
        "-i", str(url),
        "-vn",
        "-ac", "1",
        "-ar", str(sample_rate),
        "-f", "wav",
        str(temporary_path),
    ]

    try:
        result = subprocess.run(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            timeout=timeout_seconds,
        )

        if result.returncode != 0:
            error_message = result.stderr.decode("utf-8", errors="ignore").strip()
            raise RuntimeError(error_message or "ffmpeg failed to decode the audio.")

        if not temporary_path.exists() or temporary_path.stat().st_size == 0:
            raise ValueError("Downloaded audio file is empty.")

        temporary_path.replace(destination_path)

    finally:
        if temporary_path.exists():
            temporary_path.unlink(missing_ok=True)


def download_row(row: dict) -> str:
    """Download audio for one row. Returns 'downloaded', 'skipped_cached', or 'failed'."""
    row_id = row.get(ID_COLUMN)
    audio_url = row.get(AUDIO_URL_COLUMN)

    try:
        if pd.isna(audio_url) or not str(audio_url).strip():
            raise ValueError("Audio URL is missing.")

        audio_path = AUDIO_DIR / f"{row_id}.wav"

        if audio_path.exists() and audio_path.stat().st_size > 0:
            return "skipped_cached"

        download_audio(str(audio_url).strip(), audio_path)
        return "downloaded"

    except Exception as error:
        print(f"[DOWNLOAD FAILED] {ID_COLUMN}={row_id}: {error}")
        return "failed"

In [ ]:
recordings = pd.read_csv(RECORDINGS_CSV)

print(f"Rows to check: {len(recordings)}")
print(f"Workers:       {MAX_WORKERS}")

download_statuses = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [
        executor.submit(download_row, row.to_dict())
        for _, row in recordings.iterrows()
    ]

    for future in tqdm(as_completed(futures), total=len(futures), desc="Downloading audio", unit="file"):
        download_statuses.append(future.result())

status_counts = pd.Series(download_statuses).value_counts().to_dict()
print("\nDownload stage finished.")
print(f"Status counts: {status_counts}")

## Stage 3 — Classify audio → `audio_speech_labels.csv`

In [ ]:
_thread_local = threading.local()


def get_vad_model():
    if not hasattr(_thread_local, "model"):
        _thread_local.model = load_silero_vad()
    return _thread_local.model


def classify_audio(audio_path: Path) -> bool:
    """Return True if the audio at `audio_path` is majority speech."""
    sample_rate, samples = wavfile.read(str(audio_path))

    if samples.ndim > 1:
        samples = samples.mean(axis=1)

    if np.issubdtype(samples.dtype, np.integer):
        max_value = float(np.iinfo(samples.dtype).max)
        samples = samples.astype(np.float32) / max_value
    else:
        samples = samples.astype(np.float32)

    waveform = torch.from_numpy(samples)

    total_duration_seconds = waveform.numel() / sample_rate

    speech_segments = get_speech_timestamps(
        waveform,
        get_vad_model(),
        sampling_rate=sample_rate,
        return_seconds=True,
        min_speech_duration_ms=MIN_SPEECH_DURATION_MS,
        min_silence_duration_ms=MIN_SILENCE_DURATION_MS,
        speech_pad_ms=SPEECH_PAD_MS,
    )

    speech_duration_seconds = sum(
        max(0.0, float(segment["end"]) - float(segment["start"]))
        for segment in speech_segments
    )

    if total_duration_seconds > 0:
        speech_ratio = speech_duration_seconds / total_duration_seconds
    else:
        speech_ratio = 0.0

    return speech_ratio >= SPEECH_RATIO_THRESHOLD


def classify_row(row: dict):
    """Classify one row's downloaded audio. Returns the DB row + is_speech, or None to skip it."""
    row_id = row.get(ID_COLUMN)
    audio_path = AUDIO_DIR / f"{row_id}.wav"

    if not audio_path.exists() or audio_path.stat().st_size == 0:
        print(f"[CLASSIFY SKIPPED] {ID_COLUMN}={row_id}: audio not downloaded.")
        return None

    try:
        result = dict(row)
        result["is_speech"] = classify_audio(audio_path)
        return result

    except Exception as error:
        print(f"[CLASSIFY FAILED] {ID_COLUMN}={row_id}: {error}")
        return None

In [ ]:
def load_existing_results() -> pd.DataFrame:
    if not OUTPUT_CSV.exists():
        return pd.DataFrame()

    try:
        return pd.read_csv(OUTPUT_CSV)
    except Exception as error:
        print(f"Could not read existing CSV: {error}")
        return pd.DataFrame()


def get_processed_ids(existing_results: pd.DataFrame) -> set:
    if existing_results.empty or ID_COLUMN not in existing_results.columns:
        return set()
    return set(existing_results[ID_COLUMN].astype(str).tolist())


def save_results(existing_results: pd.DataFrame, new_results: list) -> pd.DataFrame:
    new_dataframe = pd.DataFrame(new_results)

    if existing_results.empty:
        combined_dataframe = new_dataframe
    else:
        combined_dataframe = pd.concat([existing_results, new_dataframe], ignore_index=True)

    combined_dataframe.to_csv(OUTPUT_CSV, index=False)
    return combined_dataframe

In [ ]:
recordings = pd.read_csv(RECORDINGS_CSV)

existing_results = load_existing_results()
processed_ids = get_processed_ids(existing_results)

rows_to_process = recordings[~recordings[ID_COLUMN].astype(str).isin(processed_ids)]

print(f"Already processed: {len(processed_ids)}")
print(f"Remaining rows:    {len(rows_to_process)}")
print(f"Workers:           {MAX_WORKERS}")

new_results = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [
        executor.submit(classify_row, row.to_dict())
        for _, row in rows_to_process.iterrows()
    ]

    for future in tqdm(as_completed(futures), total=len(futures), desc="Classifying audio", unit="file"):
        result = future.result()

        if result is not None:
            new_results.append(result)

        if new_results and len(new_results) % SAVE_EVERY == 0:
            existing_results = save_results(existing_results, new_results)
            new_results = []

if new_results:
    existing_results = save_results(existing_results, new_results)

print("\nClassification stage finished.")
print(f"Results saved to: {OUTPUT_CSV}")